# Gate 2 · CNN zoo on Colab GPU — wheeze (W) vs fine crackle (FC)

Pipeline that trains **several convolutional networks** on the SPRSound W/FC events, compares them with **leave-one-year-out cross-validation**
(each year 2023, 2024 and 2025 is held out in turn; 2022, which holds most of the data, is always part of the training set; the official split is the fold 2025 = train 2022–2024, test 2025) and exports the best one to ONNX.

1. Download SPRSound and extract the W / FC events as log-mel images (1 × 64 × 95) — same preprocessing as experiment e05.
2. Train every network of the zoo with several seeds on the GPU (mixed precision, GPU-side augmentation).
   Optional condition `pretrain`: 5 epochs on synthetic events (e05 `synth_v2.npz`) before fine-tuning on real data.
3. Rank the networks by their mean **validation AUC** over the folds (real validation patients, never the held-out year); report the held-out-year results per fold (AUC, precision / recall / F1, confusion matrices).
4. Export the best network (seed ensemble, trained for the official fold 2025) to `best_model.onnx`.

**Running it from VS Code (Colab extension):** open this notebook, *Select Kernel → Colab* and pick a GPU runtime (T4 is enough). Everything runs on the Colab machine:
data, results and checkpoints live in `/content`, which is erased when the runtime is recycled, so download the results at the end (last cell).
The notebook only needs `pip`, `git` and internet on the Colab machine; it does not need your repository.

**So the GPU session does not end before the run is finished** (Colab limits and idle disconnections cannot be switched off, and keep-alive tricks are against Colab's rules, so the run is built to tolerate them):
- `TIME_BUDGET_MIN` makes the loop stop cleanly *before* the limit; the comparison, report and ONNX cells then use what has finished. Set it below your session limit (free ≈ 3–4 h of GPU in practice, Pro ≈ 12 h or more).
- Models run from cheap to expensive and each one finishes all its folds before the next starts, so a stop always leaves complete models.
- A checkpoint is written after every model. If the runtime disconnects, reconnect, run the options + setup + data cells and the training cell again: it resumes where it stopped.
- Keep the VS Code window open and the machine awake while it trains; use the `smoke` preset first to measure the time of one model.

Set `PRESET` in the next cell: `smoke` (minutes, checks that everything works) → `long` (first serious comparison) → `full`.
For the `pretrain` condition build `synth_v2.npz` locally with `build_synth_mels.py` (see the e05 README), upload it somewhere the Colab machine can download from
(Google Drive share link → `SYNTH_URL`) or place it on the Colab machine and set `SYNTH_NPZ`. Without it only the `real` condition runs.

In [ ]:
# ── Options ──────────────────────────────────────────────────────────────────────────────────────────
from pathlib import Path

PRESET = "smoke"        # "smoke" (a few minutes) -> "long" (first serious run) -> "full"
ALL_MODELS = ["cnn_baseline", "cnn_se", "cnn_wide", "crnn", "resnet_small", "cnn_deep", "efficientnet_b0", "resnet18"]   # cheap -> expensive
PRESETS = {
    # folds = held-out years (2022 holds most of the data and is never held out); seeds are averaged into one ensemble per model
    "smoke": dict(FOLDS=["2025"], SEEDS=[0], MAX_EPOCHS=5, MODELS=["cnn_baseline", "cnn_se", "resnet_small"]),
    "long": dict(FOLDS=["2023", "2024", "2025"], SEEDS=[0, 1], MAX_EPOCHS=30, MODELS=ALL_MODELS),
    "full": dict(FOLDS=["2023", "2024", "2025"], SEEDS=[0, 1, 2], MAX_EPOCHS=40, MODELS=ALL_MODELS),
}
FOLDS, SEEDS, MODELS, MAX_EPOCHS = (PRESETS[PRESET][k] for k in ("FOLDS", "SEEDS", "MODELS", "MAX_EPOCHS"))
OFFICIAL_FOLD = "2025"                          # fold used for the report and the ONNX export (must be in FOLDS)
TIME_BUDGET_MIN = 150                           # the training loop stops cleanly before exceeding this (keep it below your Colab session limit)

DATA_ROOT = Path("/content/data/raw_data")      # train/{wav,json}, test/{wav,json}; downloaded below if missing
OUT_DIR = Path("/content/outputs")              # results and checkpoint (on the Colab machine)
SYNTH_NPZ = "/content/synth_v2.npz"             # synthetic mel set made by build_synth_mels.py; used if the file exists
SYNTH_URL = ""                                  # optional: Google Drive share link of synth_v2.npz, downloaded to SYNTH_NPZ with gdown
SAVE_TO_DRIVE = False                           # optional: also keep the checkpoint / results on Google Drive (may not work in the VS Code extension)
DRIVE_OUT = Path("/content/drive/MyDrive/biocas_gate2_zoo")

PATIENCE, BATCH = 5, 64
LR, WEIGHT_DECAY = 3e-4, 1e-3                   # fine-tuning / real-only training
PRE_EPOCHS, PRE_LR, SYNTH_RATIO = 5, 1e-3, 1.0  # synthetic pretraining
THRESHOLD = 0.5

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "soundfile", "onnx", "onnxruntime", "gdown"], check=False)

import json, time, warnings
from math import gcd
import numpy as np
import pandas as pd
import torch, torch.nn as nn
import matplotlib.pyplot as plt
import soundfile as sf
from joblib import Parallel, delayed
from scipy import signal
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, balanced_accuracy_score
from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "(no GPU: Runtime > Change runtime type)")
OUT_DIR.mkdir(parents=True, exist_ok=True)
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    except Exception as e:                      # the VS Code extension may not support the Drive mount
        print("Drive not available, results stay in", OUT_DIR, "->", e)
        SAVE_TO_DRIVE = False
if SYNTH_URL and not Path(SYNTH_NPZ).exists():
    import gdown
    gdown.download(SYNTH_URL, SYNTH_NPZ, quiet=False, fuzzy=True)
CONDITIONS = ["real"] + (["pretrain"] if Path(SYNTH_NPZ).exists() else [])
print(f"preset {PRESET}: folds {FOLDS}, seeds {SEEDS}, {len(MODELS)} models, conditions {CONDITIONS}, max {MAX_EPOCHS} epochs, budget {TIME_BUDGET_MIN} min")

## 1 · Data: download SPRSound and organise it as `train/` (2022–2024) and `test/` (2025)

In [ ]:
import shutil

def download_sprsound(root: Path):
    """Clone SPRSound and copy the recordings to root/{train,test}/{wav,json}, prefixing the file names with the year (as get_data.sh)."""
    work = Path("/content/SPRSound")
    if not work.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/SJTU-YONGFU-RESEARCH-GRP/SPRSound.git", str(work)], check=True)
    sources = [("BioCAS2022/train2022_wav", "BioCAS2022/train2022_json", "train", "2022"),
               ("BioCAS2022/test2022_wav", "BioCAS2022/test2022_json", "train", "2022"),
               ("BioCAS2023/test2023_wav", "BioCAS2023/test2023_json", "train", "2023"),
               ("BioCAS2024/test2024_wav", "BioCAS2024/test2024_json", "train", "2024"),
               ("BioCAS2025/test2025_wav", "BioCAS2025/test2025_json", "test", "2025")]
    for wav_dir, json_dir, split, year in sources:
        if not (work / wav_dir).is_dir():
            print("not found, skipped:", wav_dir)
            continue
        (root / split / "wav").mkdir(parents=True, exist_ok=True)
        (root / split / "json").mkdir(parents=True, exist_ok=True)
        jsons = {p.name: p for p in (work / json_dir).rglob("*.json")}
        n = 0
        for wav in sorted((work / wav_dir).glob("*.wav")):
            js = jsons.get(wav.stem + ".json")
            if js is None:
                continue
            shutil.copy2(wav, root / split / "wav" / f"{year}__{wav.name}")
            shutil.copy2(js, root / split / "json" / f"{year}__{js.name}")
            n += 1
        print(f"{split:5s} {year}: {n} recordings")

if not list((DATA_ROOT / "train" / "wav").glob("*.wav")):
    download_sprsound(DATA_ROOT)
print({s: len(list((DATA_ROOT / s / "wav").glob("*.wav"))) for s in ("train", "test")})

## 2 · Real events → log-mel

Each W / FC event: first channel → 4 kHz → event cut → tiled / centre-cropped to 3 s → STFT (512, hop 128) → 64 mel filters (50–2000 Hz) → dB → per-event standardisation.
Patient keys are float strings (`"40638274.0"`) like in the synthetic set, so synthetic events of validation patients can be excluded.

In [ ]:
SR, WIN_S, N_FFT, HOP, N_MELS, FMIN, FMAX = 4000, 3.0, 512, 128, 64, 50.0, 2000.0
CLASSES = ["W", "FC"]
EVENT_TYPES = {"wheeze": 0, "fine crackle": 1}

def mel_filterbank():
    freqs = np.fft.rfftfreq(N_FFT, 1 / SR)
    mel = lambda f: 2595 * np.log10(1 + f / 700)
    inv = lambda m: 700 * (10 ** (m / 2595) - 1)
    pts = inv(np.linspace(mel(FMIN), mel(FMAX), N_MELS + 2))
    fb = np.zeros((N_MELS, len(freqs)))
    for i in range(N_MELS):
        lo, c, hi = pts[i], pts[i + 1], pts[i + 2]
        fb[i] = np.maximum(0, np.minimum((freqs - lo) / (c - lo), (hi - freqs) / (hi - c)))
    return fb

FB = mel_filterbank()

def to_mel(seg):
    n = int(WIN_S * SR)
    w = seg[(len(seg) - n) // 2:(len(seg) - n) // 2 + n] if len(seg) >= n else np.tile(seg, int(np.ceil(n / len(seg))))[:n]
    _, _, z = signal.stft(w, nperseg=N_FFT, noverlap=N_FFT - HOP, window="hann", boundary="even", padded=True)
    d = 10 * np.log10(np.maximum(FB @ (np.abs(z) ** 2), 1e-10))
    return ((d - d.mean()) / (d.std() + 1e-6)).astype(np.float32)

def recording_events(wav, js, year, patient):
    ann = json.loads(Path(js).read_text(encoding="utf-8"))
    ev = []
    for e in ann.get("event_annotation", []):
        label = EVENT_TYPES.get(str(e.get("type", "")).strip().lower())
        try:
            s, t = int(e["start"]), int(e["end"])
        except (KeyError, ValueError):
            continue
        if label is not None and t - s >= 20:
            ev.append((label, s, t))
    if not ev:
        return []
    x, sr = sf.read(str(wav), dtype="float32", always_2d=True)
    x = x[:, 0]
    if sr != SR:
        g = gcd(int(sr), SR)
        x = signal.resample_poly(x, SR // g, sr // g).astype(np.float32)
    out = []
    for label, s, t in ev:
        seg = x[int(s * SR / 1000):int(t * SR / 1000)]
        if len(seg) >= 64:
            out.append((to_mel(seg), label, year, patient))
    return out

def load_real_events(root):
    jobs = []
    for split in ("train", "test"):
        for wav in sorted((root / split / "wav").glob("*.wav")):
            js = root / split / "json" / f"{wav.stem}.json"
            year, rest = wav.stem.split("__", 1)
            pid = rest.split("_")[0]
            if js.exists():
                jobs.append((wav, js, year, str(float(pid)) if pid.isdigit() else "nan"))
    rows = [r for rs in Parallel(n_jobs=-1)(delayed(recording_events)(*j) for j in jobs) for r in rs]
    X = np.stack([r[0] for r in rows])
    y, year, patient = (np.array([r[i] for r in rows]) for i in (1, 2, 3))
    return X, y, year, patient

cache = OUT_DIR / "real_events.npz"
if cache.exists():
    z = np.load(cache, allow_pickle=True); X, y, year, patient = z["X"], z["y"], z["year"], z["patient"]
else:
    X, y, year, patient = load_real_events(DATA_ROOT)
    np.savez(cache, X=X, y=y, year=year, patient=patient)
tab = pd.crosstab(year, np.array(CLASSES)[y]).rename_axis("year")
print(f"{len(y)} events"); display(tab)

synth = None
if "pretrain" in CONDITIONS:
    z = np.load(SYNTH_NPZ, allow_pickle=True)
    synth = {"X": z["mel2000"], "y": z["y"].astype(int), "year": z["year"].astype(str), "patient": z["patient"].astype(str)}
    print("synthetic events:", len(synth["y"]))

## 3 · Model zoo

All networks take the (batch, 1, 64, 95) log-mel and return one logit (FC); `sigmoid` = p(FC).

| name | idea |
|---|---|
| `cnn_baseline` | e03 / e05 best CNN: 4 × (Conv3×3-BN-ReLU-MaxPool) 16-32-64-128, GAP, MLP head (≈ 0.1 M params) |
| `cnn_wide` | same layout with 32-64-128-256 channels |
| `cnn_deep` | VGG-style: two convolutions per block, 5 blocks |
| `cnn_se` | baseline + squeeze-and-excitation channel attention in every block |
| `resnet_small` | small ResNet (residual blocks 16-32-64-128) |
| `crnn` | 3 conv blocks + bidirectional GRU over the time axis |
| `resnet18` | torchvision ResNet-18 (1 input channel, random init) |
| `efficientnet_b0` | torchvision EfficientNet-B0 (1 input channel, random init) |

In [ ]:
import torchvision

def conv_bn(c_in, c_out):
    return [nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True)]

class SE(nn.Module):
    def __init__(self, c, r=4):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(c, max(c // r, 4)), nn.ReLU(), nn.Linear(max(c // r, 4), c), nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x.mean((2, 3)))[:, :, None, None]

def head(c, drop=0.3):
    return nn.Sequential(nn.Dropout(drop), nn.Linear(c, 64), nn.ReLU(), nn.Linear(64, 1))

class PlainCNN(nn.Module):
    """Blocks of `reps` × (Conv3×3-BN-ReLU) + optional SE + MaxPool, then global average pooling and an MLP head."""
    def __init__(self, ch=(16, 32, 64, 128), reps=1, se=False):
        super().__init__()
        layers, c_in = [], 1
        for c in ch:
            for r in range(reps):
                layers += conv_bn(c_in if r == 0 else c, c)
            layers += ([SE(c)] if se else []) + [nn.MaxPool2d(2)]
            c_in = c
        self.conv, self.head = nn.Sequential(*layers), head(ch[-1])
    def forward(self, x):
        return self.head(self.conv(x).mean((2, 3))).squeeze(1)

class BasicBlock(nn.Module):
    def __init__(self, c_in, c_out, stride):
        super().__init__()
        self.body = nn.Sequential(nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
                                  nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False), nn.BatchNorm2d(c_out))
        self.skip = nn.Identity() if stride == 1 and c_in == c_out else nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride, bias=False), nn.BatchNorm2d(c_out))
    def forward(self, x):
        return torch.relu(self.body(x) + self.skip(x))

class ResNetSmall(nn.Module):
    def __init__(self, ch=(16, 32, 64, 128)):
        super().__init__()
        layers, c_in = [nn.Sequential(*conv_bn(1, ch[0]))], ch[0]
        for i, c in enumerate(ch):
            layers.append(BasicBlock(c_in, c, 1 if i == 0 else 2)); c_in = c
        self.body, self.head = nn.Sequential(*layers), head(ch[-1])
    def forward(self, x):
        return self.head(self.body(x).mean((2, 3))).squeeze(1)

class CRNN(nn.Module):
    def __init__(self, ch=(16, 32, 64), hidden=96):
        super().__init__()
        layers, c_in = [], 1
        for i, c in enumerate(ch):
            layers += conv_bn(c_in, c) + [nn.MaxPool2d(2 if i < 2 else (2, 1))]
            c_in = c
        self.conv = nn.Sequential(*layers)
        self.rnn = nn.GRU(ch[-1] * (N_MELS // 8), hidden, batch_first=True, bidirectional=True)
        self.head = head(2 * hidden)
    def forward(self, x):
        h = self.conv(x)                                   # (B, C, F', T')
        h = h.permute(0, 3, 1, 2).flatten(2)               # (B, T', C·F')
        return self.head(self.rnn(h)[0].mean(1)).squeeze(1)

def make_resnet18():
    m = torchvision.models.resnet18(weights=None, num_classes=1)
    m.conv1 = nn.Conv2d(1, 64, 3, 1, 1, bias=False); m.maxpool = nn.Identity()    # small input: keep resolution
    return Squeeze(m)

def make_efficientnet_b0():
    m = torchvision.models.efficientnet_b0(weights=None, num_classes=1)
    m.features[0][0] = nn.Conv2d(1, 32, 3, 1, 1, bias=False)
    return Squeeze(m)

class Squeeze(nn.Module):
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, x):
        return self.m(x).squeeze(1)

ZOO = {
    "cnn_baseline": lambda: PlainCNN((16, 32, 64, 128)),
    "cnn_wide": lambda: PlainCNN((32, 64, 128, 256)),
    "cnn_deep": lambda: PlainCNN((16, 32, 64, 128, 128), reps=2),
    "cnn_se": lambda: PlainCNN((16, 32, 64, 128), se=True),
    "resnet_small": lambda: ResNetSmall(),
    "crnn": lambda: CRNN(),
    "resnet18": make_resnet18,
    "efficientnet_b0": make_efficientnet_b0,
}
for name in MODELS:
    m = ZOO[name]()
    print(f"{name:16s} {sum(p.numel() for p in m.parameters()) / 1e6:6.2f} M params, output {tuple(m(torch.zeros(2, 1, 64, 95)).shape)}")

## 4 · Training and evaluation

For each (fold, condition, model, seed): the patients of the training years are split 85 / 15 (by patient) into train / validation; the held-out year of the fold is never used for training or selection.
`pretrain` = 5 epochs on synthetic events only (synthetic events of the held-out year or of validation patients are dropped), then fine-tuning on real events only.
The loss is class-weighted BCE; the best epoch is chosen by validation AUC (patience `PATIENCE`).

In [ ]:
def augment(x):
    """GPU SpecAugment-like: circular time shift, one frequency mask (≤ 8 bins), one time mask (≤ 10 frames), per sample."""
    B, _, F, T = x.shape
    dev = x.device
    shift = torch.randint(0, T, (B,), device=dev)
    idx = (torch.arange(T, device=dev)[None] - shift[:, None]) % T
    x = x.gather(3, idx[:, None, None, :].expand(B, 1, F, T))
    f = torch.randint(0, 9, (B,), device=dev); f0 = (torch.rand(B, device=dev) * (F - f)).long()
    ar = torch.arange(F, device=dev)[None]
    x = x.masked_fill(((ar >= f0[:, None]) & (ar < (f0 + f)[:, None]))[:, None, :, None], 0)
    t = torch.randint(0, 11, (B,), device=dev); t0 = (torch.rand(B, device=dev) * (T - t)).long()
    ar = torch.arange(T, device=dev)[None]
    return x.masked_fill(((ar >= t0[:, None]) & (ar < (t0 + t)[:, None]))[:, None, None, :], 0)

@torch.no_grad()
def predict(model, Xg, bs=512):
    model.eval()
    out = []
    for i in range(0, len(Xg), bs):
        with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
            out.append(torch.sigmoid(model(Xg[i:i + bs]).float()))
    return torch.cat(out).cpu().numpy()

def run_epochs(model, opt, loss_fn, Xg, yg, n_epochs, scaler, on_epoch=None):
    for ep in range(n_epochs):
        model.train()
        perm = torch.randperm(len(Xg), device=DEVICE)
        for i in range(0, len(perm), BATCH):
            b = perm[i:i + BATCH]
            if len(b) < 4:
                continue
            with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
                loss = loss_fn(model(augment(Xg[b])).float(), yg[b])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        if on_epoch is not None and on_epoch(ep):
            break

def train_one(model_name, condition, seed, held_out):
    tr = year != held_out
    Xtr, ytr, gtr = X[tr], y[tr], patient[tr]
    tr_i, va_i = next(GroupShuffleSplit(1, test_size=0.15, random_state=seed).split(Xtr, ytr, gtr))
    torch.manual_seed(seed); np.random.seed(seed)
    to_g = lambda a: torch.from_numpy(np.ascontiguousarray(a))[:, None].to(DEVICE)
    Xa, ya = to_g(Xtr[tr_i]), torch.from_numpy(ytr[tr_i].astype(np.float32)).to(DEVICE)
    Xv, yv = to_g(Xtr[va_i]), ytr[va_i]
    Xe = to_g(X[~tr])
    model = ZOO[model_name]().to(DEVICE)
    scaler = torch.amp.GradScaler(enabled=DEVICE.type == "cuda")

    if condition == "pretrain":
        ok = (synth["year"] != held_out) & ~np.isin(synth["patient"], np.unique(gtr[va_i]))
        n_per = int(round(SYNTH_RATIO * max((ytr[tr_i] == 0).sum(), (ytr[tr_i] == 1).sum())))
        srng = np.random.default_rng(seed + 1000)
        pick = np.concatenate([srng.choice(np.flatnonzero(ok & (synth["y"] == c)), size=min(n_per, int((ok & (synth["y"] == c)).sum())), replace=False)
                               for c in (0, 1)])
        Xs, ys = to_g(synth["X"][pick]), torch.from_numpy(synth["y"][pick].astype(np.float32)).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=PRE_LR, weight_decay=WEIGHT_DECAY)
        run_epochs(model, opt, nn.BCEWithLogitsLoss(), Xs, ys, PRE_EPOCHS, scaler)
        del Xs, ys

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    pos_w = torch.tensor(float((ytr[tr_i] == 0).sum() / max((ytr[tr_i] == 1).sum(), 1)), device=DEVICE)
    state = {"best": -1.0, "bad": 0, "weights": None}

    def on_epoch(ep):
        auc = roc_auc_score(yv, predict(model, Xv))
        if auc > state["best"] + 1e-4:
            state.update(best=auc, bad=0, weights={k: v.detach().clone() for k, v in model.state_dict().items()})
        else:
            state["bad"] += 1
        return state["bad"] >= PATIENCE

    run_epochs(model, opt, nn.BCEWithLogitsLoss(pos_weight=pos_w), Xa, ya, MAX_EPOCHS, scaler, on_epoch)
    model.load_state_dict(state["weights"])
    return model, state["best"], predict(model, Xe)

runs = {}          # (fold, condition, model) -> {"val_auc": [...], "p": [...], "states": [...], "seconds": ...}
y_fold = {f: y[year == f] for f in FOLDS}

In [ ]:
CKPT = OUT_DIR / "runs_checkpoint.pt"
signature = dict(PRESET=PRESET, FOLDS=FOLDS, SEEDS=SEEDS, MAX_EPOCHS=MAX_EPOCHS, CONDITIONS=CONDITIONS, LR=LR, PRE_EPOCHS=PRE_EPOCHS)
if SAVE_TO_DRIVE and (DRIVE_OUT / CKPT.name).exists() and not CKPT.exists():
    shutil.copy2(DRIVE_OUT / CKPT.name, CKPT)
if CKPT.exists():                               # resume after a disconnection / interrupted run (same settings only)
    saved = torch.load(CKPT, weights_only=False)
    if saved["signature"] == signature:
        runs = saved["runs"]; print(f"resumed {len(runs)} finished runs from {CKPT}")

fold_order = [OFFICIAL_FOLD] + [f for f in FOLDS if f != OFFICIAL_FOLD]
jobs = [(f, c, m) for m in MODELS for c in CONDITIONS for f in fold_order]       # a model finishes all its folds before the next one starts
total, t_start, n_new, stopped = len(jobs), time.time(), 0, False
slowest = {}                                    # seconds of the slowest run so far, per model, to avoid starting a run that cannot finish
for fold, condition, name in jobs:
    key = (fold, condition, name)
    if key in runs:
        continue
    if (time.time() - t_start) / 60 + slowest.get(name, 0) / 60 > TIME_BUDGET_MIN:
        stopped = True; break
    rec = {"val_auc": [], "p": [], "states": [], "seconds": 0.0}
    for seed in SEEDS:
        t0 = time.time()
        model, val_auc, p = train_one(name, condition, seed, fold)
        rec["val_auc"].append(val_auc); rec["p"].append(p)
        if fold == OFFICIAL_FOLD:                  # weights are only needed for the ONNX export
            rec["states"].append({k: v.cpu() for k, v in model.state_dict().items()})
        rec["seconds"] += time.time() - t0
        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    runs[key] = rec; n_new += 1
    slowest[name] = max(slowest.get(name, 0), rec["seconds"])
    torch.save({"signature": signature, "runs": runs}, CKPT)
    if SAVE_TO_DRIVE:
        shutil.copy2(CKPT, DRIVE_OUT / CKPT.name)
    eta = (time.time() - t_start) / n_new * (total - len(runs)) / 60
    print(f"[{len(runs)}/{total}] {name:16s} {condition:8s} fold {fold} | val AUC {np.mean(rec['val_auc']):.3f} | "
          f"held-out AUC {roc_auc_score(y_fold[fold], np.mean(rec['p'], 0)):.3f} | {rec['seconds']:.0f} s | ~{eta:.0f} min left", flush=True)
if stopped:
    print(f"\nTime budget of {TIME_BUDGET_MIN} min reached: {len(runs)}/{total} runs finished. Run this cell again (after raising TIME_BUDGET_MIN or "
          "in a new session) to resume, or continue with the comparison below, which uses only the models with all folds finished.")

## 5 · Cross-validation comparison (ranked by mean validation AUC)

In [ ]:
def summarize(y_true, p, thr=THRESHOLD):
    pred = (p >= thr).astype(int)
    r = classification_report(y_true, pred, labels=[0, 1], target_names=CLASSES, output_dict=True, zero_division=0)
    return {"test_auc": roc_auc_score(y_true, p), "balanced_acc": balanced_accuracy_score(y_true, pred), "accuracy": r["accuracy"],
            "precision_W": r["W"]["precision"], "recall_W": r["W"]["recall"], "f1_W": r["W"]["f1-score"],
            "precision_FC": r["FC"]["precision"], "recall_FC": r["FC"]["recall"], "f1_FC": r["FC"]["f1-score"],
            "macro_f1": r["macro avg"]["f1-score"], "weighted_f1": r["weighted avg"]["f1-score"]}

rows = []
for (fold, condition, name), rec in runs.items():
    rows.append({"fold": fold, "condition": condition, "model": name, "val_auc": np.mean(rec["val_auc"]),
                 **summarize(y_fold[fold], np.mean(rec["p"], 0)), "train_seconds": rec["seconds"]})
per_fold = pd.DataFrame(rows)
done = per_fold.groupby(["condition", "model"])["fold"].nunique()
complete = done[done == len(FOLDS)].index                                   # only (condition, model) with every fold finished are ranked
if len(complete) < len(done):
    print("Not ranked (unfinished folds):", [f"{c}/{m}" for (c, m) in done.index.difference(complete)])
per_fold = per_fold.set_index(["condition", "model"]).loc[complete].reset_index()
assert len(per_fold), "no model has finished all folds: run the training cell again"
per_fold.to_csv(OUT_DIR / "cv_per_fold.csv", index=False)

metric_cols = [c for c in per_fold.columns if c not in ("fold", "condition", "model", "train_seconds")]
cv = per_fold.groupby(["condition", "model"])[metric_cols].mean()
cv["test_auc_std"] = per_fold.groupby(["condition", "model"])["test_auc"].std()      # spread between held-out years
cv["params_M"] = [sum(p.numel() for p in ZOO[m]().parameters()) / 1e6 for _, m in cv.index]
cv["train_minutes"] = per_fold.groupby(["condition", "model"])["train_seconds"].sum() / 60
cv = cv.sort_values("val_auc", ascending=False)
cv.to_csv(OUT_DIR / "cv_summary.csv")
print(f"Mean over the folds {FOLDS} (ranked by validation AUC; test_* = held-out year)")
display(cv.round(3))

print("Held-out AUC per fold")
display(per_fold.pivot_table(index=["condition", "model"], columns="fold", values="test_auc").loc[cv.index].round(3))

fig, ax = plt.subplots(figsize=(9, 0.5 * len(cv) + 1.5))
labels = [f"{c} · {m}" for c, m in cv.index]
ax.barh(labels[::-1], cv["test_auc"][::-1], xerr=cv["test_auc_std"][::-1], color="#4c78a8", capsize=2)
ax.set_xlim(0.5, 1); ax.set_xlabel("held-out AUC, mean ± std over folds (seed ensemble)")
plt.tight_layout(); plt.savefig(OUT_DIR / "cv_comparison.png", dpi=120); plt.show()

### Best model

The best (condition, model) is chosen by the **mean validation AUC** (validation = real patients inside the training years), so no held-out year takes part in the choice.
Below: its confusion matrix and precision / recall / F1 for every held-out year; the official split is `OFFICIAL_FOLD`.

In [ ]:
key = cv.index[0]                                   # (condition, model)
print(f"Best by validation AUC: {key}  (mean val AUC {cv.iloc[0]['val_auc']:.3f}, mean held-out AUC {cv.iloc[0]['test_auc']:.3f})\n")

fig, axs = plt.subplots(1, len(FOLDS), figsize=(4.2 * len(FOLDS), 4.0))
axs = np.atleast_1d(axs)
metrics_out = {"model": key[1], "condition": key[0], "threshold": THRESHOLD, "folds": {}}
for ax, fold in zip(axs, FOLDS):
    p = np.mean(runs[(fold, *key)]["p"], 0)
    pred = (p >= THRESHOLD).astype(int)
    cm = confusion_matrix(y_fold[fold], pred, labels=[0, 1])
    rep = classification_report(y_fold[fold], pred, labels=[0, 1], target_names=CLASSES, output_dict=True, zero_division=0)
    metrics_out["folds"][fold] = {"confusion_matrix_rows_true_cols_pred": cm.tolist(), "auc": roc_auc_score(y_fold[fold], p), "report": rep}
    cmn = cm / cm.sum(1, keepdims=True)
    ax.imshow(cmn, vmin=0, vmax=1, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]}\n{cmn[i, j]:.1%}", ha="center", va="center", color="white" if cmn[i, j] > .5 else "black")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1]); ax.set_xticklabels(CLASSES); ax.set_yticklabels(CLASSES)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"held-out {fold} · AUC {metrics_out['folds'][fold]['auc']:.3f}", fontsize=10)
plt.tight_layout(); plt.savefig(OUT_DIR / "best_confusion_matrices.png", dpi=120); plt.show()
(OUT_DIR / "best_metrics.json").write_text(json.dumps(metrics_out, indent=2))

print(f"Official split (held-out {OFFICIAL_FOLD}):")
p_off = np.mean(runs[(OFFICIAL_FOLD, *key)]["p"], 0)
print(classification_report(y_fold[OFFICIAL_FOLD], (p_off >= THRESHOLD).astype(int), labels=[0, 1], target_names=CLASSES, zero_division=0, digits=3))
display(per_fold[(per_fold.condition == key[0]) & (per_fold.model == key[1])].set_index("fold")
        [["val_auc", "test_auc", "balanced_acc", "precision_W", "recall_W", "f1_W", "precision_FC", "recall_FC", "f1_FC", "macro_f1"]].round(3))

## 6 · Export the best model (seed ensemble of the official fold) to ONNX

In [ ]:
import onnx, onnxruntime as ort

class Ensemble(nn.Module):
    def __init__(self, models):
        super().__init__(); self.models = nn.ModuleList(models)
    def forward(self, mel):
        return torch.stack([torch.sigmoid(m(mel)) for m in self.models]).mean(0)

members = []
for st in runs[(OFFICIAL_FOLD, *key)]["states"]:
    m = ZOO[key[1]](); m.load_state_dict(st); members.append(m.eval())
ens = Ensemble(members).eval()                                  # CPU copy for a portable export
onnx_path = OUT_DIR / "best_model.onnx"
torch.onnx.export(ens, torch.zeros(1, 1, 64, 95), str(onnx_path), input_names=["mel"], output_names=["p_fc"],
                  dynamic_axes={"mel": {0: "batch"}, "p_fc": {0: "batch"}}, opset_version=17, dynamo=False)
onnx.checker.check_model(onnx.load(str(onnx_path)))
xs = X[year == OFFICIAL_FOLD][:256][:, None]
with torch.no_grad():
    ref = ens(torch.from_numpy(xs)).numpy()
out = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"]).run(None, {"mel": xs})[0]
print(f"{onnx_path} ({onnx_path.stat().st_size / 1024:.0f} KB); max |ONNX - PyTorch| = {np.abs(out - ref).max():.1e}")

In [ ]:
# Pack the results. They live on the Colab machine (/content) and disappear when the runtime is recycled.
keep = ["best_model.onnx", "best_metrics.json", "best_confusion_matrices.png", "cv_summary.csv", "cv_per_fold.csv", "cv_comparison.png"]
pack = Path("/content/gate2_zoo_results"); pack.mkdir(exist_ok=True)
for f in keep:
    shutil.copy2(OUT_DIR / f, pack / f)
zip_path = shutil.make_archive(str(pack), "zip", pack)
print("results packed in", zip_path)
if SAVE_TO_DRIVE:
    shutil.copy2(zip_path, DRIVE_OUT / Path(zip_path).name); print("copied to", DRIVE_OUT)
try:
    from google.colab import files
    files.download(zip_path)                    # works in the browser version of Colab
except Exception:
    print("Download the zip from the Colab machine (VS Code: Colab server file view -> right-click -> Download), or use SAVE_TO_DRIVE = True.")